# Lab: Credit Card Fraud Detection with Decision Trees and SVM

*This notebook demonstrates a complete data science workflow for a classification task, from data collection to model evaluation, using a real-world imbalanced dataset.*

---

## Stage 1: Business Understanding
Let's start by understanding the problem from a business perspective.

* **Scenario:** We are data scientists at a financial institution. Our task is to build a model that can identify fraudulent credit card transactions in real-time.
* **Business Objective:** To proactively detect and flag fraudulent transactions to minimize financial losses for both the institution and its customers. A successful model will accurately identify fraudulent activity while minimizing the number of legitimate transactions that are incorrectly flagged (false positives).

---

## Stage 2: Analytic Approach
With the business problem defined, we select an analytic approach.

* **Problem Framing:** The goal is to classify each tracsaction as either fraudulent (1) or legitimate (0). This is a **binary classification** problem.
* **Candidate Models:** We will evaluate two different but powerful classification algorithms:
    1. **Decision Tree Classifies:** Chosen for its high interpretability. We can easily inspect the rules it learns to understand *why* it flags a transaction.
    2. **Support Vector Machine (SVM):** Chosen for its effectiveness in high-dimensional spaces, which is relevant here due to the numerous PCA-transformed features. We will use `LinearSVC`, a fast and scalable implementation of SVM.
* **Evaluation Metric:** Because the dataset is **highly imbalanced** (very few fraudulent transactions), accuracy is a misleading metric. A model that always predicts "legitimate" would have >99% accuracy but be useless. Therefore, we will use the **Area Under the Receiver Operating Characteristics Curve (ROC-AUC)** score. This model evaluates the model's ability to distinguish between the positive and negative classes across all possible probability thresholds, making it ideal for imbalanced data.

---

## Stage 3: Data Requirements
To build our model, we need a dataset of credit card transactions where each transaction is labeled as fraudulent or legitimate. The dataset should contain features that might be predictive of fraud. The provided dataset contains anonymized features (`V1` to `V28`) resulting from a PCA transformation, along with `Time` and `Amount`.

---

## Stage 4: Data Collection
The data is available from a public URl. To ensure our project is robust and reproducible, we will first download the data to a local `./data` directory and then load it from there.

### 4.1. Importing Libraries and Setup
Let's start with importing the libraries we'll use and configure them as needed.

In [6]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score

# Configurations
plt.style.use('fivethirtyeight')
pd.set_option('display.max_columns', None)

### 4.2. Executing Data Collection and Storing Raw Data
Next, let's write and run the code that will execute the data collection and store the raw data in the `./data` directory.

In [7]:
# Define the local data directory and file path
data_dir = Path("./data")
file_path = data_dir / "creditcard.csv"
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/creditcard.csv"

# Create the data directory if it doesn't exist
if not data_dir.exists():
    data_dir.mkdir(parents=True)
    print(f"Directory '{data_dir}' created.")
    
# Download the file if it doesn't already exist
if not file_path.exists():
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # Raise an exception for bad status codes
        with open(file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Data successfully downloaded and saved to '{file_path}'.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading data: {e}")
else:
    print(f"Data file already exists at '{file_path}'.")

Directory 'data' created.
Data successfully downloaded and saved to 'data/creditcard.csv'.
